In [1]:
import pandas as pd
import os

In [4]:
file_names = [
    'Ethiopia_clean.csv',
    'kenya-clean.csv',
    'nigeria_clean.csv',
    'Sudan_clean.csv',
    'tanzania_clean.csv'
]

all_countries_df = pd.DataFrame()

for file_name in file_names:
    file_path = os.path.join('/content/', file_name)
    try:
        df = pd.read_csv(file_path)
        # Extract country name from file_name (e.g., 'Ethiopia' from 'Ethiopia_clean.csv')
        country_name = file_name.replace('_clean.csv', '').replace('-clean.csv', '').capitalize()
        df['Country'] = country_name
        all_countries_df = pd.concat([all_countries_df, df], ignore_index=True)
    except FileNotFoundError:
        print(f"Warning: {file_path} not found. Skipping.")

print(f"Total rows in combined DataFrame: {len(all_countries_df)}")
display(all_countries_df.head())

Total rows in combined DataFrame: 20540


,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,...,T2M_Zscore,T2M_MAX_Zscore,T2M_MIN_Zscore,PRECTOTCORR_Zscore,RH2M_Zscore,WS2M_Zscore,WS2M_MAX_Zscore,is_outlier,Country,Date
0,2015,1,11.73,22.75,3.44,19.31,0.0,41.79,2.73,5.07,...,-2.286045,-0.163269,-2.603645,-0.577866,-1.806604,1.088524,1.330345,False,Ethiopia,NaN
1,2015,2,12.30,24.01,4.09,19.92,0.0,33.29,2.39,4.19,...,-1.985701,0.294724,-2.354311,-0.577866,-2.383500,0.595061,0.547137,False,Ethiopia,NaN
2,2015,3,12.49,24.17,3.97,20.20,0.0,33.83,1.77,2.76,...,-1.885586,0.352882,-2.400342,-0.577866,-2.346850,-0.304782,-0.725576,False,Ethiopia,NaN
3,2015,4,14.08,23.78,6.90,16.88,0.0,38.84,0.87,1.28,...,-1.047782,0.211122,-1.276418,-0.577866,-2.006821,-1.611007,-2.042790,False,Ethiopia,NaN
4,2015,5,14.06,23.15,7.32,15.83,0.0,47.07,1.34,2.14,...,-1.058320,-0.017874,-1.115310,-0.577866,-1.448251,-0.928867,-1.277382,False,Ethiopia,NaN


# Temperature Trend Comparison

In [7]:
import numpy as np
import plotly.express as px

# Ensure 'YEAR' and 'DOY' are integers
all_countries_df['YEAR'] = all_countries_df['YEAR'].astype(int)
all_countries_df['DOY'] = all_countries_df['DOY'].astype(int)

# Create a proper 'Date' column from 'YEAR' and 'DOY'
# The 'Date' column currently has NaN, so we will regenerate it
all_countries_df['Date'] = pd.to_datetime(all_countries_df['YEAR'].astype(str) + '-' + all_countries_df['DOY'].astype(str), format='%Y-%j')

# Extract month for monthly average calculation
all_countries_df['Month'] = all_countries_df['Date'].dt.month

# Calculate monthly average T2M for each country
monthly_avg_t2m = all_countries_df.groupby(['Country', 'Date'])['T2M'].mean().reset_index()

# Create a line plot for monthly average T2M
fig = px.line(
    monthly_avg_t2m,
    x='Date',
    y='T2M',
    color='Country',
    title='Monthly Average Temperature (T2M) Trend for All Countries (2015-2026)',
    labels={'T2M': 'Average Temperature (C)', 'Date': 'Date'},
    hover_name='Country'
)

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Average Temperature (°C)',
    legend_title='Country'
)

fig.show()

In [8]:
# Summary table comparing mean, median, and standard deviation of T2M across countries
t2m_summary = all_countries_df.groupby('Country')['T2M'].agg(['mean', 'median', 'std']).reset_index()
t2m_summary.columns = ['Country', 'Mean T2M', 'Median T2M', 'Std Dev T2M']
display(t2m_summary)

,Country,Mean T2M,Median T2M,Std Dev T2M
0,Ethiopia,16.068500,16.04,1.898050
1,Kenya,20.427600,20.36,1.440824
2,Nigeria,26.656928,26.82,1.123335
3,Sudan,28.759007,29.16,4.681305
4,Tanzania,26.802422,26.99,1.325388


# Precipitation Variability Comparison


In [9]:
import plotly.express as px

# Create side-by-side boxplots for PRECTOTCORR across countries
fig = px.box(
    all_countries_df,
    x='Country',
    y='PRECTOTCORR',
    title='Precipitation Variability (PRECTOTCORR) Across Countries',
    labels={'PRECTOTCORR': 'Precipitation (mm/day)', 'Country': 'Country'},
    color='Country' # Differentiate boxes by color
)

fig.update_layout(
    xaxis_title='Country',
    yaxis_title='Precipitation (mm/day)',
    showlegend=False # Legend is redundant with colored boxes and x-axis labels
)

fig.show()

In [10]:
prectotcorr_summary = all_countries_df.groupby('Country')['PRECTOTCORR'].agg(['mean', 'median', 'std']).reset_index()
prectotcorr_summary.columns = ['Country', 'Mean PRECTOTCORR', 'Median PRECTOTCORR', 'Std Dev PRECTOTCORR']
display(prectotcorr_summary)

,Country,Mean PRECTOTCORR,Median PRECTOTCORR,Std Dev PRECTOTCORR
0,Ethiopia,3.633795,0.82,6.289061
1,Kenya,1.468162,0.38,3.180228
2,Nigeria,4.213914,1.84,7.266742
3,Sudan,0.643875,0.00,3.057672
4,Tanzania,3.740256,0.64,8.003947


# Extreme Event Frequency

In [11]:
# Identify extreme heat days (T2M_MAX > 35°C)
extreme_heat_days = all_countries_df[all_countries_df['T2M_MAX'] > 35]

# Group by Country and YEAR, then count the number of extreme heat days
extreme_heat_frequency = extreme_heat_days.groupby(['Country', 'YEAR']).size().reset_index(name='Num_Extreme_Heat_Days')

# Display the results
display(extreme_heat_frequency)

,Country,YEAR,Num_Extreme_Heat_Days
0,Sudan,2015,280
1,Sudan,2016,252
2,Sudan,2017,266
3,Sudan,2018,248
4,Sudan,2019,251
5,Sudan,2020,195
6,Sudan,2021,212
7,Sudan,2022,202
8,Sudan,2023,262
9,Sudan,2024,223


it appears that only Sudan has recorded days where the maximum temperature (T2M_MAX) exceeded 35°C in the combined dataset. This means that for the other countries, the T2M_MAX values did not go above 35°C, or did so rarely, based on the available data from 2015-2026.

In [12]:
import pandas as pd
import plotly.express as px

# Define a dry day: PRECTOTCORR < 1 mm
all_countries_df['is_dry_day'] = all_countries_df['PRECTOTCORR'] < 1

# Sort by country, year, and date to ensure correct consecutive calculation
df_sorted = all_countries_df.sort_values(by=['Country', 'YEAR', 'Date']).reset_index(drop=True)

# Create a helper column to identify changes in dry/wet status within each country-year group
# A new 'block' starts when 'is_dry_day' changes or Country/YEAR changes
group_change = (df_sorted['is_dry_day'] != df_sorted['is_dry_day'].shift(1)) | \
               (df_sorted['Country'] != df_sorted['Country'].shift(1)) | \
               (df_sorted['YEAR'] != df_sorted['YEAR'].shift(1))

# Assign a unique ID to each consecutive block
df_sorted['block_id'] = group_change.cumsum()

# Calculate the length of each block
block_lengths = df_sorted.groupby('block_id').size().reset_index(name='block_length')
df_sorted = df_sorted.merge(block_lengths, on='block_id', how='left')

# Filter for dry blocks and get the maximum length per country-year
max_consecutive_dry_days = df_sorted[df_sorted['is_dry_day'] == True].groupby(['Country', 'YEAR'])['block_length'].max().reset_index(name='Max_Consecutive_Dry_Days')

# Create all possible Country-YEAR combinations to ensure all are represented, even if no dry days
all_years = all_countries_df['YEAR'].unique()
all_countries = all_countries_df['Country'].unique()
country_year_combinations = pd.DataFrame([(c, y) for c in all_countries for y in all_years], columns=['Country', 'YEAR'])

consecutive_dry_days_full = pd.merge(country_year_combinations, max_consecutive_dry_days, on=['Country', 'YEAR'], how='left').fillna(0)

print("DataFrame showing maximum consecutive dry days per year:")
display(consecutive_dry_days_full.head())

DataFrame showing maximum consecutive dry days per year:


,Country,YEAR,Max_Consecutive_Dry_Days
0,Ethiopia,2015,25
1,Ethiopia,2016,35
2,Ethiopia,2017,43
3,Ethiopia,2018,35
4,Ethiopia,2019,46


In [13]:
# Visualize the maximum consecutive dry days per year for each country
fig = px.bar(
    consecutive_dry_days_full,
    x='YEAR',
    y='Max_Consecutive_Dry_Days',
    color='Country',
    barmode='group', # Group bars by year
    title='Maximum Consecutive Dry Days Per Year Across Countries (PRECTOTCORR < 1mm)',
    labels={'YEAR': 'Year', 'Max_Consecutive_Dry_Days': 'Max Consecutive Dry Days'},
    height=600
)

fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Max Consecutive Dry Days',
    legend_title='Country'
)

fig.show()

# Statistical Testing

In [16]:
from scipy import stats

t2m_per_country = []
for country in all_countries_df['Country'].unique():
    t2m_per_country.append(all_countries_df[all_countries_df['Country'] == country]['T2M'].dropna())

f_statistic_anova, p_value_anova = stats.f_oneway(*t2m_per_country)

print(f"One-Way ANOVA F-statistic: {f_statistic_anova:.2f}")
print(f"One-Way ANOVA P-value: {p_value_anova:.3e}")

One-Way ANOVA F-statistic: 18938.75
One-Way ANOVA P-value: 0.000e+00


In [17]:
h_statistic_kw, p_value_kw = stats.kruskal(*t2m_per_country)

print(f"Kruskal-Wallis H-statistic: {h_statistic_kw:.2f}")
print(f"Kruskal-Wallis P-value: {p_value_kw:.3e}")

Kruskal-Wallis H-statistic: 15392.99
Kruskal-Wallis P-value: 0.000e+00


### Interpretation of Statistical Tests

Both the One-Way ANOVA and Kruskal-Wallis H-test were performed to compare the `T2M` (temperature) values across the five countries (Ethiopia, Kenya, Nigeria, Sudan, Tanzania).

*   **One-Way ANOVA P-value**: `{{p_value_anova}}`
*   **Kruskal-Wallis P-value**: `{{p_value_kw}}`

Given the extremely low p-values for both tests (significantly less than typical significance levels like 0.05), we can conclude the following:

*   **Statistical Significance**: There are statistically significant differences in the mean (ANOVA) and median (Kruskal-Wallis) `T2M` values between at least some of the countries. This suggests that the temperature distributions are not the same across all five countries.

*   **Implication**: The observed differences in average temperatures between countries are unlikely to have occurred by random chance. This supports our earlier visual observations from the temperature trend plot and the summary statistics, which showed distinct temperature profiles for each country.

# **Vulnerability** Ranking & Key Observations

In [18]:
# 1. Aggregate Extreme Heat Days
extreme_heat_sum = extreme_heat_frequency.groupby('Country')['Num_Extreme_Heat_Days'].sum().reset_index()
extreme_heat_sum.columns = ['Country', 'Total_Extreme_Heat_Days']

# 2. Aggregate Average Max Consecutive Dry Days
avg_consecutive_dry = consecutive_dry_days_full.groupby('Country')['Max_Consecutive_Dry_Days'].mean().reset_index()
avg_consecutive_dry.columns = ['Country', 'Avg_Max_Consecutive_Dry_Days']

# 3. Merge all relevant summary statistics into a single DataFrame
vulnerability_df = t2m_summary[['Country', 'Mean T2M', 'Std Dev T2M']].copy()
vulnerability_df = vulnerability_df.merge(prectotcorr_summary[['Country', 'Mean PRECTOTCORR', 'Std Dev PRECTOTCORR']], on='Country')
vulnerability_df = vulnerability_df.merge(extreme_heat_sum, on='Country', how='left').fillna(0) # Fill NaN for countries with no extreme heat days
vulnerability_df = vulnerability_df.merge(avg_consecutive_dry, on='Country')

# Display the merged data before ranking
print("Combined Metrics for Vulnerability Assessment:")
display(vulnerability_df)

Combined Metrics for Vulnerability Assessment:


,Country,Mean T2M,Std Dev T2M,Mean PRECTOTCORR,Std Dev PRECTOTCORR,Total_Extreme_Heat_Days,Avg_Max_Consecutive_Dry_Days
0,Ethiopia,16.068500,1.898050,3.633795,6.289061,0.0,37.916667
1,Kenya,20.427600,1.440824,1.468162,3.180228,0.0,41.083333
2,Nigeria,26.656928,1.123335,4.213914,7.266742,0.0,35.833333
3,Sudan,28.759007,4.681305,0.643875,3.057672,2694.0,142.750000
4,Tanzania,26.802422,1.325388,3.740256,8.003947,0.0,40.250000


In [19]:
# 4. Rank each metric (1 = most vulnerable, 5 = least vulnerable)
# Higher values for T2M, T2M_StdDev, PRECTOTCORR_StdDev, Extreme_Heat_Days, Avg_Max_Consecutive_Dry_Days indicate higher vulnerability.
# Lower values for Mean_PRECTOTCORR indicate higher vulnerability.

vulnerability_df['Rank_Mean_T2M'] = vulnerability_df['Mean T2M'].rank(ascending=False, method='min')
vulnerability_df['Rank_Std_Dev_T2M'] = vulnerability_df['Std Dev T2M'].rank(ascending=False, method='min')
vulnerability_df['Rank_Mean_PRECTOTCORR'] = vulnerability_df['Mean PRECTOTCORR'].rank(ascending=True, method='min')
vulnerability_df['Rank_Std_Dev_PRECTOTCORR'] = vulnerability_df['Std Dev PRECTOTCORR'].rank(ascending=False, method='min')
vulnerability_df['Rank_Total_Extreme_Heat_Days'] = vulnerability_df['Total_Extreme_Heat_Days'].rank(ascending=False, method='min')
vulnerability_df['Rank_Avg_Max_Consecutive_Dry_Days'] = vulnerability_df['Avg_Max_Consecutive_Dry_Days'].rank(ascending=False, method='min')

# 5. Calculate Total Vulnerability Score (sum of ranks)
vulnerability_df['Total_Vulnerability_Score'] = vulnerability_df[[col for col in vulnerability_df.columns if col.startswith('Rank_')]].sum(axis=1)

# 6. Assign a final Vulnerability Rank based on the total score (1 = most vulnerable)
vulnerability_df['Vulnerability_Rank'] = vulnerability_df['Total_Vulnerability_Score'].rank(ascending=True, method='min').astype(int)

# 7. Sort and display the final ranking table
final_vulnerability_ranking = vulnerability_df.sort_values(by='Vulnerability_Rank')

print("\nFinal Climate Vulnerability Ranking:")
display(final_vulnerability_ranking[['Vulnerability_Rank', 'Country', 'Total_Vulnerability_Score',
                                     'Mean T2M', 'Mean PRECTOTCORR', 'Total_Extreme_Heat_Days', 'Avg_Max_Consecutive_Dry_Days']])


Final Climate Vulnerability Ranking:


,Vulnerability_Rank,Country,Total_Vulnerability_Score,Mean T2M,Mean PRECTOTCORR,Total_Extreme_Heat_Days,Avg_Max_Consecutive_Dry_Days
3,1,Sudan,10.0,28.759007,0.643875,2694.0,142.750000
4,2,Tanzania,16.0,26.802422,3.740256,0.0,40.250000
1,3,Kenya,17.0,20.427600,1.468162,0.0,41.083333
0,4,Ethiopia,19.0,16.068500,3.633795,0.0,37.916667
2,5,Nigeria,22.0,26.656928,4.213914,0.0,35.833333


## Summary of Climate Data Analysis and Vulnerability Ranking

This comprehensive analysis involved examining climate data for five African countries: Ethiopia, Kenya, Nigeria, Sudan, and Tanzania, covering the period from 2015 to 2026.

### Key Findings:

*   **Data Integration**: We successfully loaded and combined individual country CSV files into a single `all_countries_df` DataFrame, adding a `Country` column to facilitate country-specific analysis.

*   **Temperature Trend Comparison (`T2M`)**:
    *   A line plot visualized the monthly average temperatures for all countries, revealing distinct temperature profiles.
    *   Summary statistics (mean, median, standard deviation) for T2M were provided for each country.
    *   Statistical tests (One-Way ANOVA and Kruskal-Wallis H-test) confirmed highly significant differences in T2M values across the countries (p-values of 0.000e+00), indicating that temperature distributions are not uniform.

*   **Precipitation Variability Comparison (`PRECTOTCORR`)**:
    *   Boxplots illustrated the variability in precipitation across countries, showing diverse patterns.
    *   Summary statistics (mean, median, standard deviation) for PRECTOTCORR were also generated.

*   **Extreme Event Frequency - Extreme Heat**:
    *   We identified and counted days where the maximum temperature (`T2M_MAX`) exceeded 35°C.
    *   **Sudan** showed a significant number of extreme heat days (2694 total), while other countries did not consistently reach this threshold within the dataset.

*   **Extreme Event Frequency - Consecutive Dry Days**:
    *   The maximum consecutive dry days (PRECTOTCORR < 1mm) per year for each country were calculated and visualized with a bar chart.
    *   **Sudan** exhibited a much higher average of consecutive dry days (142.75 days) compared to the other nations, indicating prolonged dry periods.

### Climate Vulnerability Ranking:

Based on the aggregation and ranking of various climate metrics (mean T2M, std dev T2M, mean PRECTOTCORR, std dev PRECTOTCORR, total extreme heat days, and average maximum consecutive dry days), the countries were ranked for climate vulnerability:

1.  **Sudan (Rank 1)**: Identified as the most vulnerable, primarily due to its highest mean temperature, lowest mean precipitation, extensive extreme heat days, and longest average consecutive dry periods.
2.  **Tanzania (Rank 2)**: Shows moderate vulnerability, with relatively high mean temperature and notable precipitation variability.
3.  **Kenya (Rank 3)**: Positioned in the mid-range of vulnerability.
4.  **Ethiopia (Rank 4)**: Exhibits lower mean temperatures and moderate precipitation, contributing to a lower vulnerability ranking.
5.  **Nigeria (Rank 5)**: Ranked as the least vulnerable among the five countries based on the analyzed indicators, having moderate temperatures and the highest mean precipitation.

This analysis provides a comprehensive overview of the climate patterns and vulnerabilities in these African nations, highlighting Sudan's particularly high susceptibility to climate impacts based on the selected indicators.